### UCBとは
- 多腕バンディット問題において探索と活用のバランスを取る手法。
- 多腕バンディット問題とは、複数のスロットマシン（腕）があり、各腕を引くと確率的に報酬が得られるとき、累計報酬を最大化するにはどの腕をどの順番で引けばよいか、という問題。
    - 活用（Exploitation）：これまでの結果から最も良さそうな腕を引く
    - 探索（Exploration）：まだあまり試していない腕を引いて情報を集める
- UCBはこのトレードオフを「推定報酬 + 不確実性ボーナス」という形で数式的に解決する。

||内容|
|----|----|
|問題設定|多腕バンディット問題|
|UCBスコア|平均報酬（活用） + 不確実性ボーナス（探索）|



In [2]:
# numpyを用いたシミュレーション

import numpy as np

np.random.seed(42)

n_arms = 5
n_rounds = 1000
true_means = np.random.uniform(0, 1, n_arms)

counts  = np.zeros(n_arms)
rewards = np.zeros(n_arms)
total_reward = 0

for t in range(1, n_rounds + 1):
    ucb_values = rewards / np.maximum(counts, 1) + np.sqrt(2 * np.log(t) / np.maximum(counts, 1))
    arm = np.argmax(ucb_values)

    reward = np.random.binomial(1, true_means[arm])
    counts[arm]  += 1
    rewards[arm] += reward
    total_reward += reward

print(f"真の期待報酬（各腕）: {true_means.round(4)}")
print(f"最適な腕のインデックス: {np.argmax(true_means)}")
print(f"各腕の選択回数: {counts.astype(int)}")
print(f"総報酬: {total_reward}")

真の期待報酬（各腕）: [0.3745 0.9507 0.732  0.5987 0.156 ]
最適な腕のインデックス: 1
各腕の選択回数: [ 17 831  97  43  12]
総報酬: 893


In [3]:
# scratchでの実装


import numpy as np


class UCBBandit:
    def __init__(self, n_arms, n_rounds, random_state=None):
        self.n_arms = n_arms
        self.n_rounds = n_rounds
        self.rng = np.random.default_rng(random_state)

        self.counts  = np.zeros(n_arms)
        self.rewards = np.zeros(n_arms)
        self.history = []

    def _ucb_score(self, t):
        avg_rewards = self.rewards / np.maximum(self.counts, 1)
        bonus = np.sqrt(2 * np.log(t) / np.maximum(self.counts, 1))
        return avg_rewards + bonus

    def select_arm(self, t):
        return np.argmax(self._ucb_score(t))

    def update(self, arm, reward):
        self.counts[arm]  += 1
        self.rewards[arm] += reward

    def run(self, true_means):
        total_reward = 0
        for t in range(1, self.n_rounds + 1):
            arm    = self.select_arm(t)
            reward = self.rng.binomial(1, true_means[arm])
            self.update(arm, reward)
            total_reward += reward
            self.history.append(arm)
        return total_reward


# 実行
rng = np.random.default_rng(42)
n_arms   = 5
n_rounds = 1000
true_means = rng.uniform(0, 1, n_arms)

bandit = UCBBandit(n_arms=n_arms, n_rounds=n_rounds, random_state=42)
total_reward = bandit.run(true_means)

print(f"真の期待報酬（各腕）: {true_means.round(4)}")
print(f"最適な腕のインデックス: {np.argmax(true_means)}")
print(f"各腕の選択回数: {bandit.counts.astype(int)}")
print(f"総報酬: {total_reward}")

真の期待報酬（各腕）: [0.774  0.4389 0.8586 0.6974 0.0942]
最適な腕のインデックス: 2
各腕の選択回数: [265  50 489 178  18]
総報酬: 778


`_ucb_score` メソッド

導出した式をそのまま実装しています。`np.maximum(self.counts, 1)` はゼロ除算を防ぐためです。まだ1度も試していない腕（`counts=0`）はボーナスが非常に大きくなるため、最初の数ラウンドで全ての腕が1度ずつ試されます。

`run` メソッド

ラウンドごとにUCBスコアで腕を選択し、報酬を観測してパラメータを更新するループです。`history` に選択した腕の履歴を記録しており、後から分析に使うことができます。

報酬のシミュレーション

`rng.binomial(1, true_means[arm])` でベルヌーイ分布に従う報酬（0か1）をサンプリングしています。`true_means[arm]` が当選確率に相当します。

In [5]:
# regretの推移
# regretが理論上界に収まっていることを確認する

rng = np.random.default_rng(42)
true_means = rng.uniform(0, 1, n_arms)
mu_star = true_means.max()

bandit = UCBBandit(n_arms=n_arms, n_rounds=n_rounds, random_state=42)
bandit.run(true_means)

cumulative_regret = []
cumulative = 0
for t, arm in enumerate(bandit.history, 1):
    cumulative += mu_star - true_means[arm]
    cumulative_regret.append(cumulative)

deltas = mu_star - true_means[true_means < mu_star]
upper_bound = np.sum(8 * np.log(n_rounds) / deltas) + (1 + np.pi**2 / 3) * np.sum(deltas)

print(f"最終後悔: {cumulative_regret[-1]:.2f}")
print(f"理論上界: {upper_bound:.2f}")

最終後悔: 85.87
理論上界: 1205.74
